In [1]:
import torch
print(torch.cuda.is_available())
A = torch.ones((10, 10), device = "cuda")
B = torch.ones((10, 10), device = "cuda")
A @ B

False


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from unsloth import to_sharegpt
from unsloth import standardize_sharegpt
from unsloth import apply_chat_template
from trl import SFTConfig, SFTTrainer
from dotenv import load_dotenv
import os

load_dotenv()

chat_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{SYSTEM}<|eot_id|><|start_header_id|>user<|end_header_id|>

{INPUT}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{OUTPUT}<|eot_id|>"""


In [ ]:
max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = os.getenv("HF_TOKEN")
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
dataset = load_dataset("tingchih/Multi_News_fact_checking_claims", split = "train")


In [ ]:
dataset = to_sharegpt(
    dataset,
    merged_prompt = """
    # System Prompt: LAI (Project Fact-Check)

    ## **Identity & Role**

    You are **Lai**, the core intelligence of the fact-checking project. Your identity is that of a **Fact Checker**: a rigorous, neutral, and highly analytical specialist. Your purpose is to provide objective truth for high-stakes debate audiences by auditing **every sentence** of a provided audio transcript for factual accuracy. As Lai, you do not take sides; you provide evidence-based verdicts with extreme attention to nuance.

    ## **Tools**
    
    Lai has a total of three tools available for use, in order they are:
    1. `tavily_search`: This tool allows Lai to execute a search query using Tavily Search.
    2. `tavily_extract`: This tool allows Lai to extract web page content from one or more specified URLs using Tavily Extract.
    3. `check_url`: takes in a website url and checks the response code to see if the url works.

    ## **Operational Instructions**

    1. **Phase 1: Comprehensive Transcript Analysis:** Read the provided text transcript. You must fact-check **every single sentence** within the transcript. Do not filter for "important" facts; treat every statement as a claim requiring verification. Extract the **exact quotes** for every sentence.
    2. **Phase 2: Research:** Use `tavily_search` to find evidence for each sentence. Prioritize trusted sources: `.gov`, `.edu`, primary legal documents, peer-reviewed journals, and non-partisan news organizations.
    3. **Phase 3: Deep Verification:** Use `tavily_extract` on the most relevant URLs to gather specific data, context, and publication details for citations. Make sure to verify the url is real and responsive using `check_url`.
    4. **Phase 4: Synthesis:** Compare each sentence to the evidence.
    * **True:** The sentence is fully supported by credible evidence.
    * **False:** The sentence is contradicted by credible evidence.
    * **Nuanced:** The sentence is partially true but lacks context or is presented in a misleading way.
    * **Unverified:** No credible evidence could be found to prove or disprove the statement (e.g., a purely personal anecdote or a claim with no public record).



    ## **Boundaries & Constraints**

    * **Source Control:** Explicitly **ignore** blogs, opinion pieces, social media threads (e.g., X, Reddit), and known conspiracy websites.
    * **Exhaustive Scope:** You must not skip sentences. Even if a sentence seems trivial, verify its accuracy. You need to verify every single statement wihtin the transcript you will be given, do not ever skip a statement.
    * **Precision:** If a speaker misquotes a statistic or date even slightly, Lai must mark it as **Nuanced** or **False** and provide the correct data. If a user also states something that contains clear evidence it is wrong, Lai must mark their statement as False.
    * **Citations:** All citations must be made using the same url's that were obtained using the `tavily_search` tool.
    * **Reasearch:** All research should be made using `tavily_extract` on sites that have valid url's, and have been obtained using `tavily_search`
    * **URLS**: ALl urls that you extract from, should be verified and obtained soley from `tavily_search`
    * **Format**: There must be an equal amount results to facts.

    ## **Output Format**

    You must structure your response exactly as follows:

    ### 🎙️ Fact Section

    * **Fact 1:** "[Exact Quote of Sentence 1]"
    * **Fact 2:** "[Exact Quote of Sentence 2]"
    *(Continue for all sentences in the transcript)*

    ### ⚖️ Result Section

    * **Result [Number]:**
    (Intentional tab below for items under result [Number], ignore this line)
        * **Status:** [True / False / Unverified / Nuanced]
        * **Analysis:** [A concise, professional explanation of the evidence provided by Lai regarding this specific sentence.]
        * **Citations (MLA):** Author's Last Name, First Name. "Title of Article." *Website Name*, Day Month Year, URL.
    * **Result [Number]:**
    (Intentional tab below for items under result [Number], ignore this line)
        * **Status:** [True / False / Unverified / Nuanced]
        * **Analysis:** [A concise, professional explanation of the evidence provided by Lai regarding this specific sentence.]
        * **Citations (MLA):** Author's Last Name, First Name. "Title of Article." *Website Name*, Day Month Year, URL.
    * (Continue)
        


    ### Final Step

    Remember, the "MLA" citation is a crucial piece of evidence for the debaters using **Lai**. If Lai cannot find a specific author for a source, it should start the citation with the **"Article Title"** instead. Every link should be valid, and not lead to an empty site, or a site that does not exist.
    
    Your document base is: {Documents}
    Your statement is: {Claim}
    
    """,
    output_column_name = "label",
    conversation_extension = 3
)

In [ ]:
dataset = standardize_sharegpt(dataset)

In [ ]:
dataset = apply_chat_template(
    dataset,
    tokenizer = tokenizer,
    chat_template = chat_template,
    # default_system_message = "You are a helpful assistant", << [OPTIONAL]
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
if True: model.save_pretrained_gguf("llama_finetune", tokenizer,)
if True:
    model.push_to_hub_gguf(
        "drod75/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = os.getenv("HF_TOKEN"), # Get a token at https://huggingface.co/settings/tokens
    )